# Early-warning methodology validation

This notebook reviews the analytical prototype rather than treating its warning bands as approved risk classifications. It examines baseline coverage, one-at-a-time sensitivity, band stability and a deterministic current-period case sample.

In [1]:
from pathlib import Path
import json
import pandas as pd

validation_dir = Path('../data/processed/warnings/validation')
warning_dir = Path('../data/processed/warnings')
summary = json.loads((validation_dir / 'methodology_validation_summary.json').read_text())
scenarios = pd.read_csv(validation_dir / 'sensitivity_scenarios.csv')
stability = pd.read_csv(validation_dir / 'sensitivity_observation_stability.csv')
sample = pd.read_csv(validation_dir / 'warning_review_sample.csv')
drilldown = pd.read_csv(validation_dir / 'warning_review_sample_drilldown.csv')
display(pd.Series(summary))

methodology_id                    fca_complaint_early_warning_v1
scenario_count                                                15
feature_rows                                                5751
scenario_observation_rows                                  86265
review_sample_rows                                            25
review_drilldown_rows                                        200
baseline_priority_review_count                               352
fully_stable_observations                                   3384
validation_status                                         passed
validation_issues                                             []
dtype: object

## Completed case-review outcomes

The durable decision register has been validated against all 25 sampled cases. These outcomes test the methodology; they are not findings of firm misconduct.

In [2]:
review_results = pd.read_csv(validation_dir / 'warning_case_review_results.csv')
review_crosstab = pd.read_csv(validation_dir / 'warning_case_review_crosstab.csv')
review_summary = json.loads((validation_dir / 'warning_case_review_summary.json').read_text())
display(pd.Series(review_summary).drop('recommended_revisions'))
display(review_crosstab)
display(review_results[[
    'priority_band', 'display_firm_name', 'product_group', 'warning_score',
    'review_outcome', 'review_comment'
]])

sample_cases                                                                     25
decisions_recorded                                                               25
complete_decision_coverage                                                     True
review_outcome_counts             {'useful_monitoring_case': 7, 'insufficient_co...
priority_cases_reviewed                                                           5
coherent_priority_cases                                                           4
likely_false_positive_cases                                                       4
potential_false_negative_cases                                                    1
recommended_methodology_status                    revision_required_before_approval
dtype: object

,priority_band,coherent_priority_case,insufficient_context,likely_false_positive,no_material_signal_confirmed,potential_false_negative,useful_monitoring_case
0,insufficient_data,0,5,0,0,0,0
1,monitor,0,0,2,0,0,3
2,no_current_signal,0,0,0,4,1,0
3,priority_review,4,0,0,0,0,1
4,review,0,0,2,0,0,3


,priority_band,display_firm_name,product_group,warning_score,review_outcome,review_comment
0,priority_review,Embark Services Limited,decumulation_and_pensions,15,coherent_priority_case,"Independent context-rate, upheld-outcome and t..."
1,priority_review,Coutts & Company,decumulation_and_pensions,11,coherent_priority_case,Timeliness and upheld-outcome signals combine ...
2,priority_review,Scottish Equitable Plc,investments,11,coherent_priority_case,Persistent adverse peer positions occur across...
3,priority_review,Utility Warehouse Limited,insurance_and_pure_protection,11,coherent_priority_case,Provision and intermediation context rates are...
4,priority_review,Complete Cover Group Ltd,insurance_and_pure_protection,10,useful_monitoring_case,High provision and intermediation context rate...
5,review,ANIMAL FRIENDS INSURANCE SERVICES LIMITED,insurance_and_pure_protection,5,useful_monitoring_case,Intermediation context rate is elevated and in...
6,review,Barclays Bank Plc,home_finance,5,likely_false_positive,The 100% upheld rate and large percentage-poin...
7,review,Casualty & General Insurance Company (Europe) Ltd,insurance_and_pure_protection,5,useful_monitoring_case,Provision context rate increased 43.3% and abo...
8,review,Coventry Building Society,insurance_and_pure_protection,5,likely_false_positive,The 100% upheld rate and change are based on o...
9,review,HBOS Investment Fund Managers Limited,investments,5,useful_monitoring_case,Upheld percentage is elevated at 86.19% and in...


## Sensitivity of workload

The table compares each one-at-a-time scenario with the configured baseline. Large changes mean the operational review population depends materially on that judgement.

In [3]:
baseline_priority = int(scenarios.loc[scenarios.scenario_id.eq('baseline'), 'band_priority_review'].iloc[0])
scenario_view = scenarios.assign(
    priority_review_change=lambda frame: frame.band_priority_review - baseline_priority,
    priority_review_change_pct=lambda frame: (frame.band_priority_review - baseline_priority) / baseline_priority,
)[[
    'scenario_id', 'scenario_description', 'band_priority_review',
    'priority_review_change', 'priority_review_change_pct',
    'band_review', 'band_insufficient_data', 'mean_warning_score'
]].sort_values('priority_review_change')
display(scenario_view)

,scenario_id,scenario_description,band_priority_review,priority_review_change,priority_review_change_pct,band_review,band_insufficient_data,mean_warning_score
10,priority_score_8,Priority-review boundary increased from six to...,181,-171,-0.485795,1186,1618,1.262911
2,peer_high_95,High-peer threshold increased from 90th to 95t...,232,-120,-0.340909,876,1618,0.982090
8,over_8_weeks_20,Beyond-eight-weeks threshold increased from 10...,304,-48,-0.136364,846,1618,1.094592
14,minimum_rules_4,Minimum eligible rules increased from three to...,319,-33,-0.093750,844,2533,1.262911
4,context_growth_50,Context deterioration threshold increased from...,326,-26,-0.073864,1033,1618,1.225700
6,upheld_change_10,Upheld deterioration increased from five to te...,330,-22,-0.062500,1033,1618,1.224135
12,minimum_peer_30,Minimum peer group increased from 20 to 30,351,-1,-0.002841,1016,1620,1.260998
0,baseline,Configured v1 methodology,352,0,0.000000,1015,1618,1.262911
3,context_growth_20,Context deterioration threshold reduced from 2...,356,4,0.011364,1014,1618,1.271083
13,minimum_rules_2,Minimum eligible rules reduced from three to two,357,5,0.014205,1075,1212,1.262911


## Observation-level stability

A stable observation remains in its baseline band under all scenarios. Instability does not automatically invalidate a case, but it shows where prioritisation is driven by threshold choice.

In [4]:
stability_summary = pd.Series({
    'observations': len(stability),
    'fully_stable': int(stability.band_stability_rate.eq(1).sum()),
    'changed_in_at_least_one_scenario': int(stability.band_stability_rate.lt(1).sum()),
    'median_stability_rate': stability.band_stability_rate.median(),
    'minimum_stability_rate': stability.band_stability_rate.min(),
})
display(stability_summary)
display(stability.sort_values(['band_stability_rate', 'baseline_warning_score']).head(20))

observations                        5751.000000
fully_stable                        3384.000000
changed_in_at_least_one_scenario    2367.000000
median_stability_rate                  1.000000
minimum_stability_rate                 0.666667
dtype: float64

,firm_key,reporting_period,source_reporting_period,product_group,display_firm_name,baseline_priority_band,baseline_warning_score,minimum_scenario_score,maximum_scenario_score,distinct_scenario_bands,scenario_bands,scenarios_same_as_baseline,scenario_count,band_stability_rate
2045,FRN:178737,2023-H1,2023-01-01 to 2023-06-30,banking_and_credit_cards,Cater Allen Limited,review,5,2,9,3,monitor|priority_review|review,10,15,0.666667
457,FRN:110873,2023-H2,2023-07-01 to 2023-12-31,investments,Wesleyan Assurance Society,review,3,0,6,4,insufficient_data|no_current_signal|priority_r...,11,15,0.733333
232,FRN:110002,2025-H1,2025-01-01 to 2025-06-30,decumulation_and_pensions,Scottish Friendly Assurance Society Limited,review,5,2,8,3,monitor|priority_review|review,11,15,0.733333
410,FRN:110495,2021-H1,01-01-2021 to 30-06-2021,investments,ReAssure Limited,priority_review,6,3,6,3,insufficient_data|priority_review|review,11,15,0.733333
500,FRN:114724,2021-H1,01-01-2021 to 30-06-2021,home_finance,The Royal Bank of Scotland Plc,priority_review,6,3,6,3,insufficient_data|priority_review|review,11,15,0.733333
1260,FRN:122702,2021-H1,01-01-2021 to 30-06-2021,investments,Barclays Bank Plc,priority_review,6,3,6,3,insufficient_data|priority_review|review,11,15,0.733333
3464,FRN:311873,2025-H1,2025-01-01 to 2025-06-30,consumer_credit,ACORN INSURANCE AND FINANCIAL SERVICES LIMITED,priority_review,6,4,7,3,insufficient_data|priority_review|review,11,15,0.733333
4645,FRN:702607,2025-H1,2024-07-01 to 2025-06-30,consumer_credit,Damartex UK Limited,priority_review,6,4,6,3,insufficient_data|priority_review|review,11,15,0.733333
5197,FRN:758053,2025-H2,2025-07-01 to 2025-12-31,consumer_credit,Salary Finance Limited,priority_review,6,4,6,3,insufficient_data|priority_review|review,11,15,0.733333
357,FRN:110462,2023-H1,2023-01-01 to 2023-06-30,investments,ReAssure Life Limited,priority_review,7,4,8,2,priority_review|review,11,15,0.733333


## Current-period review sample

Five deterministic examples are selected from every band. Review should trace from the case to rule evidence, features, processed fact and original workbook. Review fields intentionally remain blank.

In [5]:
sample_columns = [
    'priority_band', 'display_firm_name', 'product_group', 'warning_score',
    'eligible_rule_count', 'triggered_rule_count', 'persistent_rule_count',
    'triggered_rule_ids', 'review_status'
]
display(sample[sample_columns])

,priority_band,display_firm_name,product_group,warning_score,eligible_rule_count,triggered_rule_count,persistent_rule_count,triggered_rule_ids,review_status
0,priority_review,Embark Services Limited,decumulation_and_pensions,15,6,5,2,CTX_PROVISION_DETERIORATION|CTX_PROVISION_PEER...,pending_case_review
1,priority_review,Coutts & Company,decumulation_and_pensions,11,6,4,1,CTX_PROVISION_DETERIORATION|OVER_8_WEEKS_HIGH|...,pending_case_review
2,priority_review,Scottish Equitable Plc,investments,11,6,3,3,CTX_PROVISION_PEER_HIGH|OVER_8_WEEKS_HIGH|UPHE...,pending_case_review
3,priority_review,Utility Warehouse Limited,insurance_and_pure_protection,11,8,3,2,CTX_INTERMEDIATION_PEER_HIGH|CTX_PROVISION_PEE...,pending_case_review
4,priority_review,Complete Cover Group Ltd,insurance_and_pure_protection,10,8,3,2,CTX_INTERMEDIATION_DETERIORATION|CTX_INTERMEDI...,pending_case_review
5,review,ANIMAL FRIENDS INSURANCE SERVICES LIMITED,insurance_and_pure_protection,5,6,2,0,CTX_INTERMEDIATION_DETERIORATION|CTX_INTERMEDI...,pending_case_review
6,review,Barclays Bank Plc,home_finance,5,5,2,0,UPHELD_DETERIORATION|UPHELD_PEER_HIGH,pending_case_review
7,review,Casualty & General Insurance Company (Europe) Ltd,insurance_and_pure_protection,5,6,2,0,CTX_PROVISION_DETERIORATION|OVER_8_WEEKS_HIGH,pending_case_review
8,review,Coventry Building Society,insurance_and_pure_protection,5,4,2,0,UPHELD_DETERIORATION|UPHELD_PEER_HIGH,pending_case_review
9,review,HBOS Investment Fund Managers Limited,investments,5,6,2,0,UPHELD_DETERIORATION|UPHELD_PEER_HIGH,pending_case_review


In [6]:
first_case = sample.iloc[0]
case_mask = (
    drilldown.firm_key.eq(first_case.firm_key)
    & drilldown.reporting_period.eq(first_case.reporting_period)
    & drilldown.source_reporting_period.eq(first_case.source_reporting_period)
    & drilldown.product_group.eq(first_case.product_group)
)
case_evidence = drilldown.loc[case_mask, [
    'rule_id', 'rule_label', 'eligible', 'triggered', 'persistent_trigger',
    'eligibility_reason', 'condition_evidence', 'base_points'
]].sort_values(['triggered', 'base_points'], ascending=False)
display(case_evidence)

,rule_id,rule_label,eligible,triggered,persistent_trigger,eligibility_reason,condition_evidence,base_points
3,CTX_PROVISION_PEER_HIGH,Provision context rate in highest peer decile,True,True,True,conditions_met,"{""context_provision_rate_peer_percentile"": 0.9...",3
5,OVER_8_WEEKS_HIGH,At least 10% closed beyond eight weeks and upp...,True,True,True,conditions_met,"{""closed_over_8_weeks_pct"": 0.6457999999999999...",3
7,UPHELD_PEER_HIGH,Upheld percentage in highest peer decile,True,True,False,conditions_met,"{""complaints_upheld_pct_peer_percentile"": 0.96...",3
2,CTX_PROVISION_DETERIORATION,Provision context rate increased at least 25%,True,True,False,conditions_met,"{""context_provision_rate_peer_percentile"": 0.9...",2
6,UPHELD_DETERIORATION,Upheld percentage increased at least five perc...,True,True,False,conditions_met,"{""change_complaints_upheld_pct"": 0.11629999999...",2
0,CTX_INTERMEDIATION_DETERIORATION,Intermediation context rate increased at least...,False,False,False,missing_required_feature,"{""context_intermediation_rate_peer_percentile""...",0
1,CTX_INTERMEDIATION_PEER_HIGH,Intermediation context rate in highest peer de...,False,False,False,missing_required_feature,"{""context_intermediation_rate_peer_percentile""...",0
4,OPENED_VOLUME_PRESSURE,Opened complaints increased at least 25% and a...,True,False,False,conditions_not_met,"{""complaints_opened_count_peer_percentile"": 0....",0


## Decision guidance

The completed case review recommends `revision_required`. Before approval, a business owner should review minimum denominator support, category-level point caps and the volume-pressure peer gate. Any revised thresholds should create a new methodology version rather than silently modifying v1.